Step 1: Dependencies and Setup

Bringing in the standard data science stack and Hugging Face tools.

In [1]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)

Step 2: Dataset Preparation

Loading the Financial PhraseBank. We lock the random seed to ensure our 80/20 train/test split is exactly the same every time we run this.

In [2]:
print("Loading dataset...")
# Pinning the revision to bypass recent Hugging Face security blocks on custom dataset scripts
dataset = load_dataset(
    "takala/financial_phrasebank",
    "sentences_75agree",
    revision="refs/pr/10"
)

dataset = dataset["train"].train_test_split(test_size=0.2, seed=42)
train_data = dataset["train"]
test_data = dataset["test"]

# Ground truth labels for evaluation later (0: negative, 1: neutral, 2: positive)
y_true = test_data["label"]

# Cast sentences to standard strings to avoid weird tokenizer crashes later
test_sentences = [str(sentence) for sentence in test_data["sentence"]]

Loading dataset...


README.md:   0%|          | 0.00/9.29k [00:00<?, ?B/s]

sentences_75agree/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B /  267kB            

sentences_75agree/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/3453 [00:00<?, ? examples/s]

Step 3: Evaluation Metrics

Financial data is usually heavily skewed toward neutral statements. Standard accuracy can be misleading, so we track Macro F1 to ensure the model actually learns the rare positive/negative classes.

In [3]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }

tep 4: Zero-Shot Baseline (General Sentiment)

Testing how a generic sentiment model (trained on movie reviews) handles financial jargon.

In [6]:
print("Evaluating Baseline (SST-2 Model)...")

# Switch to GPU if it's available
baseline_pipeline = pipeline(
    "text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=0 if torch.cuda.is_available() else -1
)

# Manually map the SST-2 output labels to our 0, 1, 2 integer format
def map_sst2_to_finbank(prediction):
    label = prediction['label']
    if label == "NEGATIVE": return 0
    if label == "POSITIVE": return 2
    return 1 # Default to neutral if it's unsure

baseline_preds = []
# Process in batches to keep the GPU from running out of memory
for p in baseline_pipeline(test_sentences, batch_size=16, truncation=True):
    baseline_preds.append(map_sst2_to_finbank(p))

baseline_metrics = compute_metrics(y_true, baseline_preds)
print(f"Baseline Metrics: {baseline_metrics}")

Evaluating Baseline (SST-2 Model)...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Baseline Metrics: {'accuracy': 0.2575976845151954, 'macro_f1': 0.25141714703012674}


Step 5: Domain-Pretrained Benchmark (FinBERT)

This is our upper ceiling. FinBERT was pre-trained on massive financial corpora.

In [7]:
print("Evaluating Upper Bound (FinBERT)...")

finbert_pipeline = pipeline(
    "text-classification",
    model="ProsusAI/finbert",
    device=0 if torch.cuda.is_available() else -1
)

def map_finbert(prediction):
    label = prediction['label']
    if label == "negative": return 0
    if label == "neutral": return 1
    if label == "positive": return 2

finbert_preds = []
for p in finbert_pipeline(test_sentences, batch_size=16, truncation=True):
    finbert_preds.append(map_finbert(p))

finbert_metrics = compute_metrics(y_true, finbert_preds)
print(f"FinBERT Metrics: {finbert_metrics}")

Evaluating Upper Bound (FinBERT)...


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

FinBERT Metrics: {'accuracy': 0.9493487698986975, 'macro_f1': 0.9385191535232308}


Step 6: Fine-Tuning DistilBERT (Our Experiment)

Taking a raw, lightweight DistilBERT model and training a new classification head purely on our training split for 3 epochs.

In [8]:
print("Fine-tuning DistilBERT on financial data...")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer([str(s) for s in examples["sentence"]], padding="max_length", truncation=True)

# Pre-tokenize the whole dataset upfront
tokenized_train = train_data.map(tokenize_function, batched=True)
tokenized_test = test_data.map(tokenize_function, batched=True)

# Tell the model to initialize a fresh head with exactly 3 output classes
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

def compute_trainer_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return compute_metrics(labels, predictions)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_trainer_metrics,
)

# Start the actual training loop
trainer.train()

# Evaluate on the unseen test set
finetuned_predictions = np.argmax(trainer.predict(tokenized_test).predictions, axis=-1)
finetuned_metrics = compute_metrics(y_true, finetuned_predictions)
print(f"Fine-Tuned Metrics: {finetuned_metrics}")

Fine-tuning DistilBERT on financial data...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

...

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,No log,0.237673,0.914616,0.892033
2,No log,0.303591,0.894356,0.877087
3,0.272971,0.238084,0.923300,0.905670


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-Tuned Metrics: {'accuracy': 0.9232995658465991, 'macro_f1': 0.9056699346405228}


Step 7: Export Visualizations

Generating the matrices and bar chart.

In [9]:
def plot_cm(y_true, y_pred, title, filename):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(6,5), dpi=300)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Neg', 'Neu', 'Pos'],
                yticklabels=['Neg', 'Neu', 'Pos'])

    plt.title(title, fontsize=14, pad=10)
    plt.ylabel('Actual', fontsize=12)
    plt.xlabel('Predicted', fontsize=12)
    plt.savefig(filename, bbox_inches='tight')
    plt.close()

# Save the three confusion matrices
plot_cm(y_true, baseline_preds, "Baseline (General Sentiment) CM", "baseline_cm.png")
plot_cm(y_true, finbert_preds, "FinBERT (Domain-Pretrained) CM", "finbert_cm.png")
plot_cm(y_true, finetuned_predictions, "Fine-Tuned DistilBERT CM", "finetuned_cm.png")

# Create the final grouped bar chart for the results section
def plot_metrics_bar_chart(baseline_m, finbert_m, finetuned_m, filename="model_comparison_barchart.png"):
    models = ['Baseline', 'FinBERT', 'Fine-Tuned (Ours)']
    accuracy = [baseline_m['accuracy'], finbert_m['accuracy'], finetuned_m['accuracy']]
    macro_f1 = [baseline_m['macro_f1'], finbert_m['macro_f1'], finetuned_m['macro_f1']]

    x = np.arange(len(models))
    width = 0.35

    fig, ax = plt.subplots(figsize=(8, 6), dpi=300)
    rects1 = ax.bar(x - width/2, accuracy, width, label='Accuracy', color='#4C72B0')
    rects2 = ax.bar(x + width/2, macro_f1, width, label='Macro F1', color='#DD8452')

    # Bump up the font sizes for poster readability
    ax.set_ylabel('Score', fontsize=14)
    ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold', pad=15)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=12)
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=12, loc='upper left')

    ax.bar_label(rects1, fmt='%.3f', padding=3, fontsize=11)
    ax.bar_label(rects2, fmt='%.3f', padding=3, fontsize=11)

    plt.savefig(filename, bbox_inches='tight')
    plt.close()

plot_metrics_bar_chart(baseline_metrics, finbert_metrics, finetuned_metrics)
print("Pipeline complete. All graphics successfully saved to disk.")

Pipeline complete. All graphics successfully saved to disk.
